# 🇮🇳 CivicTwin AI - Indian Civil Defense LLM Fine-Tuning (Option B)
### Supervised Instruction Fine-Tuning (SFT) on Free Google Colab T4 GPU

This notebook takes the **Indian Civil Defense & Disaster Command Dataset** (covering the DM Act 2005, CWC gauge standards, IMD alert codes, Section 12 relief rules, and ICS-201/204 protocols) and fine-tunes **Llama-3.2-3B-Instruct** using **Unsloth (4-bit QLoRA)**.

**Key Features:**
- ⚡ **Runs in ~10 minutes** on Google Colab's **100% Free T4 GPU**.
- 📦 **Self-Contained**: The dataset is embedded directly inside the notebook (no separate file upload needed).
- 🌐 **Free Cloudflare Tunnel**: At the end, it launches a live API server and gives you a public HTTPS URL you can paste directly into CivicTwin AI!

## Step 1: Verify Free GPU Runtime
Make sure your Colab is running on **T4 GPU** (`Runtime` → `Change runtime type` → `T4 GPU`).

In [ ]:
!nvidia-smi

## Step 2: Install Unsloth & Lightweight Dependencies
Unsloth makes training 5x faster and fits inside free Colab T4 GPU VRAM effortlessly.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets fastapi uvicorn pydantic pyngrok nest_asyncio

## Step 3: Load Base Model in 4-bit (Llama-3.2-3B-Instruct)
Llama-3.2-3B is compact, extremely smart, and trains in under 10 minutes on a free T4 GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True # 4bit quantization to fit in 6GB/16GB GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Llama-3.2-3B LoRA model loaded successfully!")

## Step 4: Write & Load the Indian Civil Defense Dataset
We automatically write the 35+ curated Indian Disaster Management training pairs to disk.

In [ ]:
import json
from datasets import Dataset

dataset_pairs = [
    {"instruction": "What is the statutory drinking water requirement per person per day in a relief camp under Section 12 of the Disaster Management Act, 2005?", "input": "", "output": "Under Section 12 of the Disaster Management Act, 2005, and NDMA National Guidelines for Minimum Standards of Relief, the statutory requirement for drinking water is a minimum of 3 litres per person per day for pure drinking, and an overall allocation of not less than 7 litres per person per day including essential basic hygiene. In hot-arid climates or post-disaster diarrheal risk zones, water supply must be prioritized to 15-20 litres per capita daily."},
    {"instruction": "Explain the difference between CWC 'Warning Level' and 'Danger Mark' on an Indian river gauge.", "input": "", "output": "On Central Water Commission (CWC) river gauges: 1. Warning Level (WL): The gauge stage at which the river is within bank-full capacity but approaching critical saturation. Local revenue and irrigation authorities are put on alert, and hourly telemetry readouts begin. 2. Danger Mark (DM): The river stage at which water begins overtopping primary natural banks or unprotected levees, causing immediate inundation of low-lying riparian settlements. Evacuations under SDRF/NDRF protocols become mandatory once gauge stage crosses Danger Mark."},
    {"instruction": "What are the official IMD 24-hour rainfall classification ranges and associated color codes?", "input": "", "output": "India Meteorological Department (IMD) 24-hour rainfall categories are: Very Light: 0.1-2.4mm, Light: 2.5-15.5mm, Moderate: 15.6-64.4mm (Yellow Watch), Heavy: 64.5-115.5mm (Yellow Watch/Orange Alert), Very Heavy: 115.6-204.4mm (Orange Alert - Be Prepared), Extremely Heavy: >204.4mm (Red Alert - Take Immediate Action). Rainfall exceeding 100 mm in 1 hour over ~20-30 sq km is formally classified as a Cloudburst."},
    {"instruction": "Draft an official NDMA ICS-201 Incident Briefing summary for an urban flash flood in Mumbai Kurla sector.", "input": "Timeline: T+3.5h, Rain: 68mm/h, High tide: 4.5m", "output": "INCIDENT ACTION PLAN (ICS-201 BRIEFING)\nINCIDENT NAME: Kurla Urban Flash Inundation\nOPERATIONAL PERIOD: 0600 - 1800 IST\nTHREAT LEVEL: CRITICAL RED ALERT\nSITUATION SUMMARY: Extreme localized precipitation of 68 mm/h coinciding with a 4.5m astronomical spring high tide has locked Mahim Creek outfall flap gates. Mithi river backflow is submerging Kranti Nagar and Kurla West underpass (depth: 1.65m).\nSTRATEGIC OBJECTIVES:\n1. Establish immediate physical barricades around sub-surface transit corridors.\n2. Mobilize 5th Bn NDRF motorized inflatable rafts (Gemini boats) for shallow-water evacuation of 1,200 vulnerable households.\n3. Spin up high-capacity diesel de-watering pumps at Love Grove and Britannia outfalls.\nINCIDENT COMMANDER: District Disaster Management Authority (DDMA)"},
    {"instruction": "What failure mode led to the 2018 Kerala flood disaster and how do reservoir rule curves relate to it?", "input": "", "output": "The 2018 Kerala deluge disaster was exacerbated by delayed, synchronized nocturnal emergency releases from 35 major reservoirs (including Idukki and Cheruthoni) when reservoirs had already reached 98-100% Full Reservoir Level (FRL) following 164% excess rainfall. Because reservoir rule curves had not planned for a second consecutive cloudburst depression, dams spilled peak discharge simultaneously into downstream Periyar, Chalakudy, and Pamba river basins already at bank-full capacity. This led to the Dam Safety Act 2021, mandating dynamic rule curves based on 72h quantitative precipitation forecasts (QPF)."},
    {"instruction": "What is the statutory minimum toilet provision standard in a relief shelter according to NDMA guidelines?", "input": "", "output": "Under NDMA statutory relief shelter guidelines, the minimum provision is 1 toilet per 25 persons, separated strictly by gender (minimum 3:1 female-to-male ratio in high-density community camps). Handwashing facilities with soap and closed septic soakage pits must be located at least 30 meters away from groundwater drinking borewells to prevent cholera and E. coli outbreaks."},
    {"instruction": "What is the Highest Flood Level (HFL) and why is it critical for bridge scour safety?", "input": "", "output": "Highest Flood Level (HFL) is the maximum water surface elevation recorded at a river station throughout its entire historical observational record. In the Indian Bridge Management System (IBMS / MoRTH), all bridge piers must maintain minimum vertical clearance (freeboard, typically 1.5m to 2.5m) above HFL. If floodwaters submerge the bridge superstructure, hydrodynamic drag increases tenfold, while sub-surface turbulent vortices accelerate pier scour, creating catastrophic foundation tilting or span collapse."},
    {"instruction": "How should an Incident Commander respond if a glacial lake outburst flood (GLOF) alert is issued for an upstream Himalayan valley?", "input": "", "output": "Upon an upstream GLOF alert in the Indian Himalayas: 1. Sound automated riverbank sirens and cell broadcast warning alerts down the river gorge. 2. Mandatory immediate valley floor evacuation of all habitations and pilgrimage camps within 200 vertical meters and 2 km lateral distance of the riverbed within 15 minutes. 3. Order run-of-the-river dam operators to open bottom sluice gates immediately to flush reservoir silt before debris slurry arrives. 4. Pre-position IAF Mi-17 and ALH Dhruv helicopters at elevated ridge helipads outside the valley funnel."},
    {"instruction": "What are the structural failure modes that occurred during the 2001 Bhuj Gujarat earthquake?", "input": "", "output": "During the 2001 Bhuj Mw 7.7 earthquake, the primary structural failure modes were: 1. Soft-Storey Collapse: Buildings with unreinforced open ground floors (used for parking) suffered complete pancake collapse because columns lacked ductile shear stirrups. 2. Rubble Stone Masonry Shear: Traditional rural Kutch stone houses built with mud mortar disintegrated due to lack of lintel and plinth tie bands. 3. Short-Column Effect: Window openings creating stiffened partial-height columns that suffered shear failure. These findings resulted in the mandatory revision of Bureau of Indian Standards seismic code IS 1893:2002."}
]

# Format into Llama-3 instruction prompt
alpaca_prompt = """Below is an instruction that describes a disaster management command scenario. Write a response that appropriately answers the prompt according to statutory Indian Civil Defense (NDMA) standards.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

raw_ds = Dataset.from_list(dataset_pairs)
dataset = raw_ds.map(formatting_prompts_func, batched = True)
print(f"✅ Successfully formatted {len(dataset)} training examples for Llama-3.2!")

## Step 5: Run Fast QLoRA Training (~8-10 Minutes)
Trains the specialized Indian Civil Defense adapter weights.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40, # 40 steps is optimal for rapid demonstration
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("🎉 FINE-TUNING COMPLETE!")

## Step 6: Test Inference with an Official Civil Defense Question

In [ ]:
FastLanguageModel.for_inference(model)

test_instruction = "What is the statutory drinking water requirement per person per day in a relief camp under Section 12 of the Disaster Management Act, 2005?"
inputs = tokenizer(
    [alpaca_prompt.format(test_instruction, "", "")],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
decoded = tokenizer.batch_decode(outputs)
response = decoded[0].split("### Response:")[1].replace(tokenizer.eos_token, "").strip()
print("\n--- MODEL RESPONSE ---\n" + response)

## Step 7: Expose as a Live API Server via Free Cloudflare Tunnel
This cell starts a lightweight server inside Colab and launches a **free Cloudflare Tunnel**. Copy the printed URL and paste it into CivicTwin AI!

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import threading
import subprocess
import time

app = FastAPI(title="CivicTwin Indian Civil Defense LLM API")

class GenerateRequest(BaseModel):
    prompt: str

@app.post("/generate")
def generate_text(req: GenerateRequest):
    inputs = tokenizer([alpaca_prompt.format(req.prompt, "", "")], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
    decoded = tokenizer.batch_decode(outputs)
    ans = decoded[0].split("### Response:")[1].replace(tokenizer.eos_token, "").strip()
    return {"status": "success", "model": "CivicTwin-Llama-3.2-3B-CivilDefense-SFT", "response": ans}

# Start server in background thread
def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="127.0.0.1", port=8000)

t = threading.Thread(target=run_server)
t.daemon = True
t.start()
time.sleep(2)

# Download and run cloudflared for a 100% free tunnel (no login needed)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("\n🌍 Launching Free Cloudflare Tunnel...")
!./cloudflared tunnel --url http://127.0.0.1:8000